# Phase 1: Step 1.1 — NOAA IBTrACS Ground Truth Pipeline
### Project: DeepCyclone / CycloneAI
**Author:** Sachin Yadav | **Basin:** North Indian Ocean (Bay of Bengal & Arabian Sea)

---

### Pipeline Objectives:
1. **Autonomous Ingestion:** Automatically downloads the official **NOAA IBTrACS** North Indian Ocean dataset (`ibtracs.NI.list.v04r00.csv`) if not found locally.
2. **Data Cleaning & Harmonization:** Cleans records across 1990–2024 and harmonizes WMO & JTWC/USA wind observations (`11,800+` fixes across `330+` cyclonic storms).
3. **IMD Category Classification:** Maps standard **India Meteorological Department (IMD)** intensity classes (Depression to Super Cyclonic Storm).
4. **Rapid Intensification (RI) Labeling:** Detects operational RI events ($\ge 30\text{ kts}$ intensification in 24 hours).
5. **Temporal Train / Val / Test Splits:** Splits without temporal leakage:
   - **Train:** 1990 – 2018 (Historical baseline)
   - **Validation:** 2019 – 2021 (Recent major cyclones: Fani, Amphan, Tauktae)
   - **Test:** 2022 – 2024 (Operational benchmarks: Biparjoy, Remal, Dana)
6. **Statistical Visualization:** Plots wind distributions, IMD category counts, and RI event ratios.

## 1. Setup & Automated Dataset Ingestion
If running in Google Colab or an environment without the local file, this cell automatically downloads the official NOAA North Indian Ocean archive (~12 MB) directly from NOAA NCEI servers in seconds.

In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# File target: North Indian Ocean IBTrACS archive
csv_filename = "ibtracs.NI.list.v04r00.csv"

# Potential local search paths
search_paths = [
    csv_filename,
    "IBTrACS.NI.list.v04r00.csv",
    f"../ai_engine/data/raw_ibtracs/{csv_filename}",
    f"ai_engine/data/raw_ibtracs/{csv_filename}",
    f"../{csv_filename}"
]

target_path = None
for p in search_paths:
    if os.path.exists(p) and os.path.getsize(p) > 100000:
        target_path = p
        break

if target_path is None:
    print(f"'{csv_filename}' not found locally. Downloading official NOAA IBTrACS dataset...")
    noaa_url = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r00/access/csv/ibtracs.NI.list.v04r00.csv"
    urllib.request.urlretrieve(noaa_url, csv_filename)
    target_path = csv_filename
    print(f"Download complete: {os.path.getsize(csv_filename) / (1024*1024):.2f} MB saved to '{csv_filename}'.")
else:
    print(f"Found existing dataset at: {target_path} ({os.path.getsize(target_path) / (1024*1024):.2f} MB)")

# Ingest CSV: Skip line 1 which contains IBTrACS unit labels
df_raw = pd.read_csv(target_path, skiprows=[1], low_memory=False)
print(f"Ingested Raw Shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns.")
df_raw[['SID', 'SEASON', 'NAME', 'ISO_TIME', 'LAT', 'LON', 'WMO_WIND']].head()

## 2. Feature Selection & Radiometric Quality Filtering (1990–2024)
We extract core meteorological fields and handle wind observations across reporting agencies (`WMO_WIND` harmonized with `USA_WIND` fallback).

In [ ]:
keep_columns = [
    'SID',          # Unique Storm ID
    'SEASON',       # Year
    'NAME',         # Cyclone Name
    'ISO_TIME',     # UTC Timestamp
    'LAT',          # Latitude (°N)
    'LON',          # Longitude (°E)
    'WMO_WIND',     # WMO Sustained Wind Speed (knots)
    'USA_WIND',     # JTWC Sustained Wind Speed (knots)
    'WMO_PRES',     # Central Pressure (hPa)
    'STORM_SPEED',  # Forward Translation Speed (km/h)
    'STORM_DIR'     # Translation Heading (degrees)
]

available_cols = [c for c in keep_columns if c in df_raw.columns]
df = df_raw[available_cols].copy()

# Convert numeric attributes safely
numeric_fields = ['SEASON', 'LAT', 'LON', 'WMO_WIND', 'USA_WIND', 'WMO_PRES', 'STORM_SPEED', 'STORM_DIR']
for field in numeric_fields:
    if field in df.columns:
        df[field] = pd.to_numeric(df[field], errors='coerce')

# Harmonize sustained wind: prioritize WMO, fallback to JTWC (USA_WIND) to preserve modern observations
if 'USA_WIND' in df.columns:
    df['WIND'] = df['WMO_WIND'].fillna(df['USA_WIND'])
else:
    df['WIND'] = df['WMO_WIND']

# Filter for modern satellite era: 1990 to 2024
df = df[(df['SEASON'] >= 1990) & (df['SEASON'] <= 2024)].copy()
df = df.dropna(subset=['LAT', 'LON', 'WIND']).reset_index(drop=True)

print("=== CLEANED IBTrACS NORTH INDIAN OCEAN DATASET ===")
print(f"Total Observations:  {len(df):,}")
print(f"Unique Named Storms: {df['SID'].nunique():,}")
print(f"Observation Seasons: {int(df['SEASON'].min())} – {int(df['SEASON'].max())}")
print(f"Wind Speed Range:    {df['WIND'].min():.1f} kts to {df['WIND'].max():.1f} kts (Mean: {df['WIND'].mean():.1f} kts)")
df.head()

## 3. Official IMD Category Classification
We classify every storm fix according to the official **India Meteorological Department (IMD)** 3-minute / 10-minute sustained wind scale.

In [ ]:
def map_imd_category(wind):
    """Maps sustained wind speed (knots) to official IMD operational classification."""
    if wind < 17:
        return 'Low Pressure Area'
    elif wind <= 27:
        return 'Depression'
    elif wind <= 33:
        return 'Deep Depression'
    elif wind <= 47:
        return 'Cyclonic Storm'
    elif wind <= 63:
        return 'Severe Cyclonic Storm'
    elif wind <= 89:
        return 'Very Severe Cyclonic Storm'
    elif wind <= 119:
        return 'Extremely Severe Cyclonic Storm'
    else:
        return 'Super Cyclonic Storm'

df['IMD_CATEGORY'] = df['WIND'].apply(map_imd_category)

print("=== IMD CATEGORY FREQUENCY BREAKDOWN ===")
cat_counts = df['IMD_CATEGORY'].value_counts()
for cat, count in cat_counts.items():
    pct = (count / len(df)) * 100
    print(f"  {cat:32s} : {count:5d} ({pct:5.1f}%)")

## 4. Rapid Intensification (RI) Labeling
Rapid Intensification is defined meteorologically as a wind speed increase of **$\ge 30\text{ knots}$ ($55.6\text{ km/h}$) within a 24-hour window**.

In [ ]:
# Ensure strict chronological order per cyclone track
df = df.sort_values(by=['SID', 'ISO_TIME']).reset_index(drop=True)

# In standard 6-hourly fixes: 4 steps ahead = +24 hours
df['WIND_24H_AHEAD'] = df.groupby('SID')['WIND'].shift(-4)
df['WIND_CHANGE_24H'] = df['WIND_24H_AHEAD'] - df['WIND']
df['RI_EVENT'] = (df['WIND_CHANGE_24H'] >= 30.0).astype(int)

ri_total = df['RI_EVENT'].sum()
eval_rows = df['WIND_CHANGE_24H'].notna().sum()
print("=== RAPID INTENSIFICATION (RI) DETECTION ===")
print(f"Total 24h Evaluated Pairs: {eval_rows:,}")
print(f"Total RI Events Detected:  {ri_total} ({ri_total / eval_rows * 100:.2f}% of evaluated steps)")

# Sample detected RI events
df[df['RI_EVENT'] == 1][['SID', 'NAME', 'ISO_TIME', 'WIND', 'WIND_24H_AHEAD', 'WIND_CHANGE_24H']].head()

## 5. Temporal Train / Validation / Test Splits
We avoid spatial and cross-track leakage by enforcing a temporal partition:
- **Train (1990 – 2018):** Primary model training
- **Validation (2019 – 2021):** Hyperparameter tuning (Fani, Amphan, Tauktae)
- **Test (2022 – 2024):** Held-out operational evaluation (Biparjoy, Remal)

In [ ]:
train_df = df[df['SEASON'] <= 2018].copy()
val_df   = df[(df['SEASON'] >= 2019) & (df['SEASON'] <= 2021)].copy()
test_df  = df[df['SEASON'] >= 2022].copy()

print("=== TEMPORAL SPLIT VERIFICATION ===")
print(f"Train (1990–2018): {len(train_df):5,d} fixes across {train_df['SID'].nunique():3d} storms ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val   (2019–2021): {len(val_df):5,d} fixes across {val_df['SID'].nunique():3d} storms ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test  (2022–2024): {len(test_df):5,d} fixes across {test_df['SID'].nunique():3d} storms ({len(test_df)/len(df)*100:.1f}%)")

# Export splits for PyTorch dataset ingestion
train_df.to_csv("train_ibtracs.csv", index=False)
val_df.to_csv("val_ibtracs.csv", index=False)
test_df.to_csv("test_ibtracs.csv", index=False)
print("\nSplits exported successfully: 'train_ibtracs.csv', 'val_ibtracs.csv', 'test_ibtracs.csv'.")

## 6. Scientific Distribution Visualizations
We plot the sustained wind speed distribution, IMD categories, and the RI distribution across the North Indian Ocean.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Maximum Sustained Wind Speed Histogram
axes[0].hist(df['WIND'], bins=25, color='#f97316', edgecolor='#18181b', alpha=0.9)
axes[0].axvline(df['WIND'].median(), color='#ef4444', linestyle='--', linewidth=2, label=f"Median: {df['WIND'].median():.0f} kts")
axes[0].set_title("North Indian Ocean Wind Speed Distribution (1990–2024)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Sustained Wind Speed (knots)", fontsize=10)
axes[0].set_ylabel("Observation Count", fontsize=10)
axes[0].grid(True, linestyle=':', alpha=0.6)
axes[0].legend(loc='upper right')

# Plot 2: IMD Category Bar Chart
category_order = [
    'Low Pressure Area', 'Depression', 'Deep Depression',
    'Cyclonic Storm', 'Severe Cyclonic Storm',
    'Very Severe Cyclonic Storm', 'Extremely Severe Cyclonic Storm',
    'Super Cyclonic Storm'
]
available_categories = [cat for cat in category_order if cat in df['IMD_CATEGORY'].values]
category_series = df['IMD_CATEGORY'].value_counts().reindex(available_categories)

bars = axes[1].barh(category_series.index, category_series.values, color='#38bdf8', edgecolor='#18181b')
axes[1].set_title("Official IMD Cyclone Category Distribution", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Number of Observations", fontsize=10)
axes[1].grid(True, linestyle=':', alpha=0.6)

# Add count labels on bars
for bar in bars:
    w = bar.get_width()
    axes[1].text(w + 30, bar.get_y() + bar.get_height()/2, f"{int(w):,}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("================================================================")
print("PHASE 1 (STEP 1.1) IBTRACS PIPELINE COMPLETED SUCCESSFULLY!")
print("================================================================")